<h1 style="background-color: #B6D0E2; color:#007acc; padding:18px; text-align:center; font-weight:bold; border-radius:8px;">
    Programare Liniară – Algoritm Simplex Primal⚙️🖥️
</h1>




***<p style="text-align: right; color:  #007acc;">Facultatea de Științe Aplicate</p>***
<p style="text-align: right; color:  #007acc;">Dedu Anișoara-Nicoleta, 1333a</p>
<p style="text-align: right; color:  #007acc;">Dumitrescu Andreea Mihaela, 1333a</p>
<p style="text-align: right; color:  #007acc;">Iliescu Daria-Gabriela, 1333a</p>
<p style="text-align: right; color:  #007acc;">Lungu Ionela-Diana, 1333a</p>


<span style="font-size:20px; font-weight:bold; color:#007acc;">
1. Importare bibliotecilor și definirea funcțiilor uzuale
</span>

In [25]:
import numpy as np
from fractions import Fraction

#Functie care transforma orice numar in fractie
def f(n):
    if abs(n) < 1e-10: return "0"                                                # Daca e aproape de zero, afisam 0
    return str(Fraction(float(n)).limit_denominator(100))                        # Transformare in fractie


# Functie pentru afisarea vectorilor sub forma de coloana (vertical)
def afiseaza_coloana(vector, nume):
    print(f"   {nume} = ")
    for val in vector:
        print(f"      ( {f(val):^5} )")                                          # Afisare centrata in paranteze

<span style="font-size:20px; font-weight:bold; color:#007acc;">
2. Standardizare (PL)=>(PLS)
</span>

In [26]:
def pregateste_forma_standard(A, b, c, semne, tip_x, opt_tip, M_val):
    m, n_init = np.array(A).shape                                              # Dimensiunile matricei initiale
    A_mat = np.array(A, dtype=float)                                         # Copie matrice A
    b_lucru = np.array(b, dtype=float)                                         # Copie vector b
    c_vec = np.array(c, dtype=float)                                           # Definim c_vec pentru a fi folosit în R1
    semne_lucru = list(semne)                                                  # Copie lista de semne

    # --- REGULA R1: Tratarea condițiilor variabilelor (x >= 0, x <= 0 sau liber) 
    #initializare
    coloane_r1 = []
    costuri_r1 = []
    mapare_var = []
    nume_var_r1 = [] 
    
    for j in range(n_init):
        if tip_x[j] == '>=0':
                                   # Variabila e deja standard 
            coloane_r1.append(A_mat[:, j])
            costuri_r1.append(c_vec[j])
            mapare_var.append({'nume': f"x{j+1}", 'original': j, 'semn': 1})
            nume_var_r1.append(f"x{j+1}")           # Salvăm numele
            
        elif tip_x[j] == '<=0':
            # Substituție x = -x' 
            # Înmulțim coloana și coeficientul din funcția obiectiv cu -1
            coloane_r1.append(A_mat[:, j] * (-1))
            costuri_r1.append(c_vec[j] * (-1))
            mapare_var.append({'nume': f"x{j+1}'", 'original': j, 'semn': -1})
            nume_var_r1.append(f"x{j+1}'")
            
        elif tip_x[j] == 'liber':
            # Substituție x = x' - x'' 
            # Variabila "liberă" se sparge în două coloane în tabel
            coloane_r1.append(A_mat[:, j]) # Coloana normală (x')
            costuri_r1.append(c_vec[j])
            mapare_var.append({'nume': f"x{j+1}'", 'original': j, 'semn': 1})
            nume_var_r1.append(f"x{j+1}'")
            
            coloane_r1.append(A_mat[:, j] * (-1)) # Coloana cu minus (x'')
            costuri_r1.append(c_vec[j] * (-1))
            mapare_var.append({'nume': f"x{j+1}''", 'original': j, 'semn': -1})
            nume_var_r1.append(f"x{j+1}''")
            
     # transformam listele înapoi în matrice de lucru pentru pasul urm
    A_lucru_r1 = np.column_stack(coloane_r1)
    C_lucru_r1 = np.array(costuri_r1)

    # Verificam conditia b >= 0 
    for i in range(m):                                                         # Parcurgem restrictiile
        if b_lucru[i] < 0:                                                     # Daca termenul liber e negativ
            b_lucru[i] *= -1                                                   # Inmultim linia cu -1
            A_lucru_r1[i] *= -1                                                # Inmultim coeficientii cu -1
            if semne_lucru[i] == '<=': semne_lucru[i] = '>='                   # Inversam semnul
            elif semne_lucru[i] == '>=': semne_lucru[i] = '<='

    # --- REGULA R2 Constructia coloanelor pentru Forma Standard
    coloane_std = A_lucru_r1.tolist()                                            # Matricea de lucru sub forma de lista
    Cj_std = C_lucru_r1.tolist()                                                         #Coeficientii functiei obiectiv f
    nume_var = nume_var_r1.copy()                                             # x1, x2, x3...
    baza_initiala = []                                                        # Retinem coloanele care intra in prima baza

    for i in range(m):                                                        # Trecem prin fiecare restrictie pentru a adauga y sau z
        if semne_lucru[i] == '<=':                                            # Daca avem <=
            col = [0]*m; col[i] = 1                                           # Variabila de ecart y
            for r in range(m): coloane_std[r].append(col[r])                  # Adaugam coloana in matrice
            Cj_std.append(0)                                                  # Coeficientul lui y in f este 0
            nume_var.append(f"y{i+1}")                                        # Numele variabilei
            baza_initiala.append(len(Cj_std) - 1)                             # Aceasta variabila formeaza baza
        elif semne_lucru[i] == '>=':                                          # Daca avem >=
            col_y = [0]*m; col_y[i] = -1                                      # Variabila de surplus y
            for r in range(m): coloane_std[r].append(col_y[r])
            Cj_std.append(0)                                                  # Coeficientul lui y in f este 0
            nume_var.append(f"y{i+1}")
            col_z = [0]*m; col_z[i] = 1                                       # Variabila artificiala z pentru baza
            for r in range(m): coloane_std[r].append(col_z[r])
            val_M = M_val if opt_tip == 'MIN' else -M_val                     # Penalizare M
            Cj_std.append(val_M)                                              # Coeficientul lui z in f este M
            nume_var.append(f"z{i+1}")
            baza_initiala.append(len(Cj_std) - 1)                             # Intra in baza
        elif semne_lucru[i] == '=':                                           # Daca avem =
            col_z = [0]*m; col_z[i] = 1                                       # Adaugam direct variabila artificiala z
            for r in range(m): coloane_std[r].append(col_z[r])
            val_M = M_val if opt_tip == 'MIN' else -M_val
            Cj_std.append(val_M)
            nume_var.append(f"z{i+1}")
            baza_initiala.append(len(Cj_std) - 1)

    # Returnam datele pregatite pentru alg Simplex
    return np.array(coloane_std, dtype=float), np.array(b_lucru, dtype=float), np.array(Cj_std, dtype=float), nume_var, baza_initiala, mapare_var

<span style="font-size:20px; font-weight:bold; color:#007acc;">
3. Tabel Simplex (Iterații+pivotare)
</span>

In [27]:
def ruleaza_iteratii_simplex(TS, XB, Cj, baza, nume_var, opt_tip):
    m = len(XB)                                                                    # Numar de restrictii
    n_tot = len(Cj)                                                                # Numar total de variabile
    nume_ai = [f"a{i+1}" for i in range(n_tot)]                                    # Etichete coloane a1, a2...
    pas = 0 # Contor tabele
    
    while pas < 1000:                                                                # Maxim 1000 tabele
        CB = Cj[baza]                                                              # Coeficientii bazei (coloana stanga)
        Z_total = np.dot(CB, XB)                                                   # Valoarea functiei f: suma(CB * XB)
        
        # Calcul Delta_j = Cj - Zj 
        deltas = [Cj[j] - np.dot(CB, TS[:, j]) for j in range(n_tot)]
        
        # Afisarea tabelului in consola 
        print(f"\n--- TABEL SIMPLEX T{pas} ---")
        print("CB\tBaza\tXB\t| " + "\t".join(nume_ai))
        for i in range(m):
            linie = f"{f(CB[i])}\t{nume_ai[baza[i]]}\t{f(XB[i])}\t| "
            linie += "\t".join([f(val) for val in TS[i]])
            print(linie)
        print("-" * 100)
        print(f"\tDj\tZ={f(Z_total)}\t| " + "\t".join([f(d) for d in deltas]))

        # Verificare optim 
        if opt_tip == 'MAX':
            if all(d <= 1e-5 for d in deltas): break                                   # Optim MAX daca toti Dj <= 0
            col_p = np.argmax(deltas)                                                  # Intra coloana cu Dj maxim (+)
        else: # MIN
            if all(d >= -1e-5 for d in deltas): break                                  # Optim MIN daca toti Dj >= 0
            col_p = np.argmin(deltas)                                                  # Intra coloana cu Dj minim (-)

        # Criteriul raportului minim pentru alegerea liniei pivot
        rapoarte = [XB[i]/TS[i, col_p] if TS[i, col_p] > 1e-10 else float('inf') for i in range(m)]
        
        lin_p = np.argmin(rapoarte)                                                    # Linia cu raportul cel mai mic
        
        # Actualizare tabel prin pivotare 
        pivot_val = TS[lin_p, col_p]                                                   # Valoare pivot
        TS[lin_p] /= pivot_val                                                         # Impartim linia la pivot
        XB[lin_p] /= pivot_val                                                         # Impartim XB la pivot
        for i in range(m):                                                             # Facem 0 pe restul coloanei pivotului
            if i != lin_p:
                multiplicator = TS[i, col_p]
                TS[i] -= multiplicator * TS[lin_p]
                XB[i] -= multiplicator * XB[lin_p]
        
        baza[lin_p] = col_p                                                           # Actualizam componenta bazei
        pas += 1                                                                      # Tabelul urmator
        
    return XB, Z_total, deltas, baza, TS                                              # Returnam rezultatele finale

<span style="font-size:20px; font-weight:bold; color:#007acc;">
4. Validarea soluției (V1, V2, V3)
</span>

In [28]:
def validare_solutie(XB_final, Z_final, deltas_final, baza_finala, TS_final, A_prim_init, b_init, c_init, mapare, nume_v, opt_t):
    print("\n" + "="*60)
    print("            VERIFICARI SI VALIDARE FINALA ")
    print("="*60)

    # 1. Verificare Criteriu Dj (Criteriul de optim)
    print(f"\nCRITERIU OPTIM ({opt_t}):")
    if opt_t == 'MAX':
        print(f"   Dj = {[f(d) for d in deltas_final]} <= 0 {'[OK]' if all(d <= 1e-5 for d in deltas_final) else '[FAIL]'}")
    else:
        print(f"   Dj = {[f(d) for d in deltas_final]} >= 0 {'[OK]' if all(d >= -1e-5 for d in deltas_final) else '[FAIL]'}")

    # 2. V1: Verificarea xj >= 0 
    print("\nV1) Verificare nenegativitate xj >= 0:")
    sol_completa = {nume_v[i]: 0.0 for i in range(len(nume_v))}                                # Cream dictionar cu toate var = 0
    for i in range(len(XB_final)): sol_completa[nume_v[baza_finala[i]]] = XB_final[i]          # Punem valorile bazei
    
    # Reconstructia x-urilor originale din variabilele de tabel (x', x'', etc.)
    x_reconstruit = np.zeros(len(c_init))
    for m_info in mapare:
        val_tabel = sol_completa[m_info['nume']]
        x_reconstruit[m_info['original']] += m_info['semn'] * val_tabel
        print(f"   {m_info['nume']}* = {f(val_tabel)} >= 0 [OK]")                             # Verificam doar variabilele x (decizie)

    # 3. V2: Verificarea valorii functiei f 
    print(f"\nV2) Verificare valoare f(PL) = {f(Z_final)}:")
    f_calculata = sum(c_init[i] * x_reconstruit[i] for i in range(len(c_init)))               # Inlocuim x in f
    print(f"   Calcul: {f(f_calculata)} == {f(Z_final)} {'[OK]' if abs(f_calculata - Z_final) < 1e-10 else '[FAIL]'}")

    # 4. V3: Verificarea relatiei XB(initial) = S * XB(final) 
    print("\nV3) Verificare relatie XB(I0) = S * XB(I_stop):")
    S_matrice = A_prim_init[:, baza_finala]                                           # Matricea S (coloanele din T0 corespunzatoare bazei finale)
    print("   Matricea S (din coloanele bazei finale din T0):")
    for r in S_matrice: print("      | " + "\t".join([f(val) for val in r]) + " |")
    afiseaza_coloana(XB_final, "XB(I_stop) final")                                    # Afisam XB optim vertical
    reconstruit = np.dot(S_matrice, XB_final)                                         # Produsul matricial S * XB_final
    afiseaza_coloana(reconstruit, "S * XB(I_stop)")                                   # Trebuie sa fie egal cu b initial
    afiseaza_coloana(b_init, "XB(I0) initial")                                        # Vectorul b de la care am plecat
    
    print(f"\n   Rezultat V3: {'[VERIFICAT]' if np.allclose(reconstruit, b_init) else '[FAIL]'}")
    print("="*60)

<span style="font-size:20px; font-weight:bold; color:#007acc;">
5. Funcția de Control (Glue)
</span>

Aceasta funcție apeleaza toate bucățile de cod de mai sus în ordine corectă

In [29]:
def rezolva_simplex_complet(A, b, c, semne, tip_x, opt_tip='MAX', M=1000):
    # Pas 1 & 2: Pregatire Forma Standard (R1 si R2)
    rezultat_pregatire = pregateste_forma_standard(A, b, c, semne, tip_x, opt_tip, M)
    TS_init, b_lucru, Cj_std, nume_v, baza_init, mapare = rezultat_pregatire
    
    b_backup = b_lucru.copy() # Copie b pentru V3
    A_prim_init = TS_init.copy() # Copie matrice T0 pentru V3
    
    # Pas 3: Executie Iteratii Simplex
    # XB_f = valorile bazei la final, Z_f = f optim, baza_f = indicii bazei la final
    rezultat_it = ruleaza_iteratii_simplex(TS_init, b_lucru.copy(), Cj_std, baza_init, nume_v, opt_tip)
    
    if rezultat_it:
        XB_f, Z_f, Dj_f, baza_f, TS_f = rezultat_it
        # Pas 4: Validare conform cerintelor
        validare_solutie(XB_f, Z_f, Dj_f, baza_f, TS_f, A_prim_init, b_backup, c, mapare, nume_v, opt_tip)
    else:
        print("Problema nu are solutie optima finita (nemarginita).")

<span style="font-size:20px; font-weight:bold; color:#007acc;">
6. Datele problemei PL (MIN/MAX)
</span>

### ***INPUT***

<span style="background-color: #B6D0E2; font-size:15px; font-weight:bold; color:blue;">
 1. Problemă seminar (MAX)
</span>

In [30]:
# --- DATE INTRARE ---

c_init = [3, 1, 1]
A_init = [[2, 2, 3], 
          [3, 1, 1], 
          [2, 1, 2]]
b_init = [12, 11, 8]
semne_init = ['<=', '<=', '<=']

# REGULA R1: Definim conditia pentru fiecare variabila x1, x2, x3
# Optiuni: '>=0' , '<=0' , 'liber'
tip_x_init = ['>=0', '>=0', '>=0'] 

tip_opt = 'MAX'

# RULARE PROGRAM
rezolva_simplex_complet(A_init, b_init, c_init, semne_init, tip_x_init, opt_tip=tip_opt)



--- TABEL SIMPLEX T0 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6
0	a4	12	| 2	2	3	1	0	0
0	a5	11	| 3	1	1	0	1	0
0	a6	8	| 2	1	2	0	0	1
----------------------------------------------------------------------------------------------------
	Dj	Z=0	| 3	1	1	0	0	0

--- TABEL SIMPLEX T1 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6
0	a4	14/3	| 0	4/3	7/3	1	-2/3	0
3	a1	11/3	| 1	1/3	1/3	0	1/3	0
0	a6	2/3	| 0	1/3	4/3	0	-2/3	1
----------------------------------------------------------------------------------------------------
	Dj	Z=11	| 0	0	0	0	-1	0

            VERIFICARI SI VALIDARE FINALA 

CRITERIU OPTIM (MAX):
   Dj = ['0', '0', '0', '0', '-1', '0'] <= 0 [OK]

V1) Verificare nenegativitate xj >= 0:
   x1* = 11/3 >= 0 [OK]
   x2* = 0 >= 0 [OK]
   x3* = 0 >= 0 [OK]

V2) Verificare valoare f(PL) = 11:
   Calcul: 11 == 11 [OK]

V3) Verificare relatie XB(I0) = S * XB(I_stop):
   Matricea S (din coloanele bazei finale din T0):
      | 1	2	0 |
      | 0	3	0 |
      | 0	2	1 |
   XB(I_stop) final = 
      ( 14/3  )
      ( 11/

<span style="background-color: #B6D0E2; font-size:15px; font-weight:bold; color:blue;">
 2. Problemă curs (MIN)
</span>

In [31]:
# Coeficientii functiei obiectiv 
c_fct = [3, -2, 7]

# Matricea coeficientilor din restrictii
A_prob = [[1, 2, -1], 
          [2, -1, 2], 
          [3, 2, -1]]

# Vectorul termenilor liberi (b)
b_prob = [3, 2, 4]

# Semnele restrictiilor (trebuie sa fie tot atatea cate linii are A)
semne_prob = ['<=', '<=', '>=']

# REGULA R1: Definim conditia pentru fiecare variabila x1, x2, x3
# Optiuni: '>=0' , '<=0' , 'liber'
tip_x_init = ['>=0', '>=0', '>=0'] 

# Tipul de optim ('MAX' sau 'MIN')
tip_opt = 'MIN'

# --- RULARE ---
rezolva_simplex_complet(A_prob, b_prob, c_fct, semne_prob, tip_x_init, opt_tip=tip_opt)


--- TABEL SIMPLEX T0 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a7
0	a4	3	| 1	2	-1	1	0	0	0
0	a5	2	| 2	-1	2	0	1	0	0
1000	a7	4	| 3	2	-1	0	0	-1	1
----------------------------------------------------------------------------------------------------
	Dj	Z=4000	| -2997	-2002	1007	0	0	1000	0

--- TABEL SIMPLEX T1 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a7
0	a4	2	| 0	5/2	-2	1	-1/2	0	0
3	a1	1	| 1	-1/2	1	0	1/2	0	0
1000	a7	1	| 0	7/2	-4	0	-3/2	-1	1
----------------------------------------------------------------------------------------------------
	Dj	Z=1003	| 0	-7001/2	4004	0	2997/2	1000	0

--- TABEL SIMPLEX T2 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a7
0	a4	9/7	| 0	0	6/7	1	4/7	5/7	-5/7
3	a1	8/7	| 1	0	3/7	0	2/7	-1/7	1/7
-2	a2	2/7	| 0	1	-8/7	0	-3/7	-2/7	2/7
----------------------------------------------------------------------------------------------------
	Dj	Z=20/7	| 0	0	24/7	0	-12/7	-1/7	7001/7

--- TABEL SIMPLEX T3 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a7
0	a5	9/4	| 0	0	3/2	7/4	1	5/4	-5/4
3	a1	1/2	| 1	0	0	-1/2	0	-

<span style="background-color: #B6D0E2; font-size:15px; font-weight:bold; color:blue;">
 3. Problema Dietei - PDF (MIN)
</span>

In [32]:
# Coeficientii functiei obiectiv 
c_fct_dieta = [0.3, 2.4, 1.3, 0.9, 2, 1.9]

# Matricea coeficientilor din restrictii
A_prob_dieta = [[110, 205, 160, 160, 420, 260], 
          [4, 32, 13, 8, 4, 14], 
          [2, 12, 54, 285, 22, 80]]

# Vectorul termenilor liberi (b)
b_prob_dieta = [2000, 55, 800]

# Semnele restrictiilor (trebuie sa fie tot atatea cate linii are A)
semne_prob_dieta = ['>=', '>=', '>=']

# REGULA R1: Definim conditia pentru fiecare variabila x1, x2, x3
# Optiuni: '>=0' , '<=0' , 'liber'
tip_x_init_dieta = ['>=0', '>=0', '>=0','>=0','>=0','>=0'] 

# Tipul de optim ('MAX' sau 'MIN')
tip_opt_dieta = 'MIN'

# --- RULARE ---
rezolva_simplex_complet(A_prob_dieta, b_prob_dieta, c_fct_dieta, semne_prob_dieta, tip_x_init_dieta, opt_tip=tip_opt_dieta)


--- TABEL SIMPLEX T0 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a7	a8	a9	a10	a11	a12
1000	a8	2000	| 110	205	160	160	420	260	-1	1	0	0	0	0
1000	a10	55	| 4	32	13	8	4	14	0	0	-1	1	0	0
1000	a12	800	| 2	12	54	285	22	80	0	0	0	0	-1	1
----------------------------------------------------------------------------------------------------
	Dj	Z=2855000	| -1159997/10	-1244988/5	-2269987/10	-4529991/10	-445998	-3539981/10	1000	0	1000	0	1000	0

--- TABEL SIMPLEX T1 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a7	a8	a9	a10	a11	a12
1000	a8	88400/57	| 6206/57	3767/19	2464/19	0	23236/57	12260/57	-1	1	0	0	32/57	-32/57
1000	a10	1855/57	| 351/89	3008/95	1091/95	0	115/34	670/57	0	0	-1	1	2/71	-2/71
9/10	a4	160/57	| 1/100	4/95	18/95	1	1/13	16/57	0	0	0	0	0	0
----------------------------------------------------------------------------------------------------
	Dj	Z=30085048/19	| -9364123/83	-14945057/65	-3388015/24	0	-37403698/91	-18827758/83	1000	0	1000	0	-10021/17	27021/17

--- TABEL SIMPLEX T2 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a

<span style="background-color: #B6D0E2; font-size:15px; font-weight:bold; color:blue;">
 5. Problema Transportului - PDF (MIN)
</span>

In [33]:
# --- DATE CORECTATE PENTRU PROBLEMA TRANSPORTULUI ---

# Coeficientii functiei obiectiv (COSTURILE)
# Trebuie să fie exact 21: 7 pentru Gary, 7 pentru Cleveland, 7 pentru Pittsburgh
c_fct_tr = [
    39, 14, 11, 14, 16, 82, 8,   # Gary -> 7 destinații
    27, 9, 12, 9, 26, 95, 17,    # Cleveland -> 7 destinații (Lipsea 27 de aici!)
    24, 14, 17, 13, 28, 99, 20   # Pittsburgh -> 7 destinații
]

# Matricea coeficientilor din restrictii (A)
# 10 rânduri (3 oferte + 7 cereri) x 21 coloane
A_prob_tr = [
    # Restricții Ofertă (Capacitate fabrici)
    [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], # Gary
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0], # Cleveland
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], # Pittsburgh
    
    # Restricții Cerere (Nevoie orașe)
    [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], # FRA
    [0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], # DET
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0], # LAN
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], # WIN
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0], # STL
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0], # FRE
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]  # LAF
]

# Vectorul termenilor liberi (b)
b_prob_tr = [1400, 2600, 2900, 900, 1200, 600, 400, 1700, 1100, 1000]

# Semnele restrictiilor (10 semne pentru cele 10 linii din A)
semne_prob_tr = ['='] * 10

# Tipul variabilelor (toate cele 21 sunt >= 0)
tip_x_init_tr = ['>=0'] * 21

# Tipul de optim
tip_opt_tr = 'MIN'

# --- RULARE ---

rezolva_simplex_complet(A_prob_tr, b_prob_tr, c_fct_tr, semne_prob_tr, tip_x_init_tr, opt_tip=tip_opt_tr)


--- TABEL SIMPLEX T0 ---
CB	Baza	XB	| a1	a2	a3	a4	a5	a6	a7	a8	a9	a10	a11	a12	a13	a14	a15	a16	a17	a18	a19	a20	a21	a22	a23	a24	a25	a26	a27	a28	a29	a30	a31
1000	a22	1400	| 1	1	1	1	1	1	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0
1000	a23	2600	| 0	0	0	0	0	0	0	1	1	1	1	1	1	1	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0
1000	a24	2900	| 0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	1	1	1	1	1	1	0	0	1	0	0	0	0	0	0	0
1000	a25	900	| 1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0
1000	a26	1200	| 0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0
1000	a27	600	| 0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1	0	0	0	0
1000	a28	400	| 0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1	0	0	0
1000	a29	1700	| 0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1	0	0
1000	a30	1100	| 0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1	0
1000	a31	1000	| 0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	1
---------------------------------------------------------------------